# Colab Inference for `v6_dinov2_lss_bev_cleaned`

Notebook для Google Colab:
- скачивает/читает kaggle-safe dataset `letiti6e/bev-yandex`
- загружает `best.pt`, `ema_best.pt`, `config.json`, `merged_cleaned.csv`, `test_matched_val200_split.npz`
- подбирает global threshold на `val`
- инферит `test`
- собирает несколько `zip`-посылок


In [ ]:
# Если нужно, можно раскомментировать
# !pip install -q kaggle


In [ ]:
import os
import gc
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'


## Paths

Ожидается, что kaggle dataset уже добавлен в Colab/Notebook environment,
а артефакты модели загружены вручную в `MODEL_DIR`.


In [ ]:
# Подправь под свою сессию Colab
DATA_ROOT = Path('/content/bev-yandex')
MODEL_DIR = Path('/content/v6_artifacts')
OUT_DIR = Path('/content/v6_infer_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_TRAIN = DATA_ROOT / 'autonomy_yandex_dataset_train'
DATA_VAL = DATA_ROOT / 'autonomy_yandex_dataset_val'
DATA_TEST = DATA_ROOT / 'autonomy_yandex_dataset_test'

BEST_PT = MODEL_DIR / 'best.pt'
EMA_BEST_PT = MODEL_DIR / 'ema_best.pt'
CONFIG_JSON = MODEL_DIR / 'config.json'
MERGED_CLEANED_CSV = MODEL_DIR / 'merged_cleaned.csv'
SPLIT_NPZ = MODEL_DIR / 'test_matched_val200_split.npz'

assert DATA_TRAIN.exists(), DATA_TRAIN
assert DATA_VAL.exists(), DATA_VAL
assert DATA_TEST.exists(), DATA_TEST
assert BEST_PT.exists(), BEST_PT
assert EMA_BEST_PT.exists(), EMA_BEST_PT
assert CONFIG_JSON.exists(), CONFIG_JSON
assert MERGED_CLEANED_CSV.exists(), MERGED_CLEANED_CSV
assert SPLIT_NPZ.exists(), SPLIT_NPZ

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
cfg_json = json.loads(CONFIG_JSON.read_text())
cfg = {
    'img_hw': tuple(cfg_json.get('img_hw', [392, 700])),
    'rover_emb_dim': int(cfg_json.get('rover_emb_dim', 8)),
    'rover_cond_dim': int(cfg_json.get('rover_cond_dim', 8)),
    'backbone_name': cfg_json.get('backbone_name', 'dinov2_vitb14'),
    'hub_repo': cfg_json.get('hub_repo', 'facebookresearch/dinov2'),
    'backbone_out_dim': int(cfg_json.get('backbone_out_dim', 768)),
    'patch_size': int(cfg_json.get('patch_size', 14)),
    'tap_layers': tuple(cfg_json.get('tap_layers', [2, 5, 8, 11])),
    'neck_dim': int(cfg_json.get('neck_dim', 128)),
    'context_dim': int(cfg_json.get('context_dim', 80)),
    'depth_bins': int(cfg_json.get('depth_bins', 24)),
    'depth_min': float(cfg_json.get('depth_min', 1.0)),
    'depth_max': float(cfg_json.get('depth_max', 80.0)),
    'world_z_min': float(cfg_json.get('world_z_min', -2.0)),
    'world_z_max': float(cfg_json.get('world_z_max', 4.5)),
    'bev_base_channels': int(cfg_json.get('bev_base_channels', 96)),
    'bev_gn_groups': int(cfg_json.get('bev_gn_groups', 8)),
    'use_amp': bool(cfg_json.get('use_amp', True)),
}
cfg


In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p

    cand = base_dir / p
    if cand.exists():
        return cand

    cand = base_dir.parent / p
    if cand.exists():
        return cand

    parts = list(p.parts)
    for anchor in ['autonomy_yandex_dataset_train', 'autonomy_yandex_dataset_val', 'autonomy_yandex_dataset_test']:
        if anchor in parts:
            i = parts.index(anchor)
            rel = Path(*parts[i + 1:])
            cand = base_dir / rel
            if cand.exists():
                return cand

    return base_dir / p.name


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def remap_kaggle_paths(df: pd.DataFrame, train_dir: Path, val_dir: Path, test_dir: Path) -> pd.DataFrame:
    df = df.copy()
    path_cols = [*CAMERA_NAMES, *INTRINSICS_NAMES, *CAR2CAM_NAMES, GT_NAME]

    def pick_root(split: str) -> Path:
        if split == 'train':
            return train_dir
        if split == 'val':
            return val_dir
        if split == 'test':
            return test_dir
        return train_dir

    def rewrite_path(p, split: str):
        if pd.isna(p):
            return p
        p = str(p)
        pp = Path(p)
        if pp.exists():
            return p

        root = pick_root(split)
        parts = Path(p).parts
        for anchor in ['autonomy_yandex_dataset_train', 'autonomy_yandex_dataset_val', 'autonomy_yandex_dataset_test']:
            if anchor in parts:
                i = parts.index(anchor)
                rel = Path(*parts[i + 1:])
                return str(root / rel)
        return str(root / Path(p).name) if not (root / p).exists() else str(root / p)

    if '__source_split' in df.columns:
        df['__data_root'] = df['__source_split'].map({
            'train': str(train_dir),
            'val': str(val_dir),
            'test': str(test_dir),
        }).fillna(str(train_dir))

    for col in path_cols:
        if col in df.columns:
            df[col] = [rewrite_path(p, split) for p, split in zip(df[col], df.get('__source_split', pd.Series(['train'] * len(df))))]
    return df


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
clean_info = pd.read_csv(MERGED_CLEANED_CSV)
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx'].tolist()
val_idx = split['val_idx'].tolist()

clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
train_info = remap_kaggle_paths(clean_info.iloc[train_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(clean_info.iloc[val_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
test_info = remap_kaggle_paths(load_info_with_root(DATA_TEST, 'test'), DATA_TRAIN, DATA_VAL, DATA_TEST)

print('train:', len(train_info), 'val:', len(val_info), 'test:', len(test_info))
print(train_info['__data_root'].value_counts())


In [ ]:
def build_rover_vocab_from_train(train_df: pd.DataFrame, min_count: int = 30, topk: int = 25):
    counts = train_df['rover'].value_counts()
    eligible = counts[counts >= min_count]
    top = eligible.head(topk)
    vocab = {'__other__': 0}
    for i, rover in enumerate(top.index.tolist(), start=1):
        vocab[rover] = i
    return vocab


def encode_rover(rover_name: str, rover_vocab: dict) -> int:
    return int(rover_vocab.get(rover_name, 0))


rover_vocab = build_rover_vocab_from_train(train_info)
print('num rover classes:', len(rover_vocab))


In [ ]:
class BEVDatasetV4Clean(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train', img_hw=(392, 700), rover_vocab=None):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        arr = np.array(canvas)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x
        K[1, 2] += pad_y

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def __getitem__(self, idx: int):
        row = self.info.iloc[idx]
        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out

        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out


In [ ]:
def gn_groups(channels: int, requested: int = 8) -> int:
    g = min(requested, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return max(g, 1)


class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=8, act=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False), nn.GroupNorm(gn_groups(out_c, groups), out_c)]
        if act:
            layers.append(nn.SiLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)


class ResidualBlock2d(nn.Module):
    def __init__(self, in_c, out_c, stride=1, groups=8):
        super().__init__()
        self.conv1 = ConvGNAct(in_c, out_c, k=3, s=stride, p=1, groups=groups, act=True)
        self.conv2 = ConvGNAct(out_c, out_c, k=3, s=1, p=1, groups=groups, act=False)
        self.skip = ConvGNAct(in_c, out_c, k=1, s=stride, p=0, groups=groups, act=False) if (stride != 1 or in_c != out_c) else nn.Identity()
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class ASPP2d(nn.Module):
    def __init__(self, in_c, out_c, rates=(1, 3, 6), groups=8):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(gn_groups(out_c, groups), out_c),
                nn.SiLU(inplace=True),
            )
            for r in rates
        ])
        self.proj = ConvGNAct(out_c * len(rates), out_c, k=1, s=1, p=0, groups=groups, act=True)
    def forward(self, x):
        return self.proj(torch.cat([b(x) for b in self.branches], dim=1))


class _DINOv2MultiScaleBackbone(nn.Module):
    def __init__(self, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, groups=8):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.tap_layers = tuple(tap_layers)
        self.vit = self._load_hub_model()
        self.laterals = nn.ModuleList([nn.Conv2d(out_dim, neck_dim, 1) for _ in self.tap_layers])
        self.fuse = nn.Sequential(
            ConvGNAct(len(self.tap_layers) * neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
            ConvGNAct(neck_dim, neck_dim, k=3, s=1, p=1, groups=groups, act=True),
        )
        self.down1 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.down2 = ConvGNAct(neck_dim, neck_dim, k=3, s=2, p=1, groups=groups, act=True)
        self.neck_out = ConvGNAct(neck_dim * 3, neck_dim, k=1, s=1, p=0, groups=groups, act=True)

    def _load_hub_model(self):
        last_err = None
        attempts = [
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ]
        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e
        raise RuntimeError(f'Failed to load DINOv2 backbone. Last error: {last_err}')

    def _reshape_tokens(self, tokens, H, W):
        B = tokens.shape[0]
        expected_tokens = (H // self.patch_size) * (W // self.patch_size)
        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        return tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()

    def _extract_intermediate(self, x):
        H, W = x.shape[-2:]
        try:
            feats = self.vit.get_intermediate_layers(x, n=list(self.tap_layers), reshape=True, return_class_token=False)
        except Exception:
            feats = self.vit.get_intermediate_layers(x, n=len(self.tap_layers), reshape=True, return_class_token=False)
        out = []
        for feat in feats:
            if isinstance(feat, (tuple, list)):
                feat = feat[0]
            if feat.ndim == 3:
                feat = self._reshape_tokens(feat, H, W)
            out.append(feat)
        return out

    def forward(self, x):
        feats = self._extract_intermediate(x)
        laterals = [proj(feat) for proj, feat in zip(self.laterals, feats)]
        p0 = self.fuse(torch.cat(laterals, dim=1))
        p1 = self.down1(p0)
        p2 = self.down2(p1)
        p1_up = F.interpolate(p1, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=p0.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.neck_out(torch.cat([p0, p1_up, p2_up], dim=1))
        return {'fused': fused}


class LSSViewTransform2D(nn.Module):
    def __init__(self, in_c, context_c, depth_bins, depth_min, depth_max, bev_h, bev_w, bev_res, x_range, y_range, z_min, z_max, groups=8):
        super().__init__()
        self.context_c = context_c
        self.depth_bins = depth_bins
        self.depth_min = float(depth_min)
        self.depth_max = float(depth_max)
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.bev_res = float(bev_res)
        self.x_range = x_range
        self.y_range = y_range
        self.z_min = float(z_min)
        self.z_max = float(z_max)
        self.depth_head = nn.Sequential(ConvGNAct(in_c, in_c, 3, 1, 1, groups, True), nn.Conv2d(in_c, depth_bins, 1))
        self.context_head = nn.Sequential(ConvGNAct(in_c, in_c, 3, 1, 1, groups, True), nn.Conv2d(in_c, context_c, 1))

    def _build_frustum(self, Hf, Wf, Hi, Wi, device, dtype):
        depths = torch.linspace(self.depth_min, self.depth_max, self.depth_bins, device=device, dtype=dtype)
        xs = (torch.arange(Wf, device=device, dtype=dtype) + 0.5) * (Wi / Wf)
        ys = (torch.arange(Hf, device=device, dtype=dtype) + 0.5) * (Hi / Hf)
        d, y, x = torch.meshgrid(depths, ys, xs, indexing='ij')
        return x, y, d

    def forward(self, feat_2d, intrinsics, car2cams, image_hw):
        B, N, C, Hf, Wf = feat_2d.shape
        Hi, Wi = image_hw
        feat_bn = feat_2d.reshape(B * N, C, Hf, Wf)
        depth_logits = self.depth_head(feat_bn).float()
        context = self.context_head(feat_bn).float()
        depth_prob = torch.softmax(depth_logits, dim=1).reshape(B, N, self.depth_bins, Hf, Wf)
        context = context.reshape(B, N, self.context_c, Hf, Wf)

        x_img, y_img, depth_vals = self._build_frustum(Hf, Wf, Hi, Wi, feat_2d.device, torch.float32)
        x_img = x_img.view(1, 1, self.depth_bins, Hf, Wf)
        y_img = y_img.view(1, 1, self.depth_bins, Hf, Wf)
        depth_vals = depth_vals.view(1, 1, self.depth_bins, Hf, Wf)

        intrinsics = intrinsics.float()
        car2cams = car2cams.float()
        cam2cars = torch.inverse(car2cams.reshape(B * N, 4, 4)).reshape(B, N, 4, 4)

        fx = intrinsics[..., 0, 0].view(B, N, 1, 1, 1)
        fy = intrinsics[..., 1, 1].view(B, N, 1, 1, 1)
        cx = intrinsics[..., 0, 2].view(B, N, 1, 1, 1)
        cy = intrinsics[..., 1, 2].view(B, N, 1, 1, 1)

        X = (x_img - cx) / fx * depth_vals
        Y = (y_img - cy) / fy * depth_vals
        Z = depth_vals.expand(B, N, -1, -1, -1)
        ones = torch.ones_like(Z)
        pts_cam = torch.stack([X, Y, Z, ones], dim=-1)
        pts_car = torch.einsum('bnij,bndhwj->bndhwi', cam2cars, pts_cam)

        world_x = pts_car[..., 0]
        world_y = pts_car[..., 1]
        world_z = pts_car[..., 2]
        x_idx = torch.floor((world_x - self.x_range[0]) / self.bev_res).long()
        y_idx = torch.floor((world_y - self.y_range[0]) / self.bev_res).long()
        valid = ((x_idx >= 0) & (x_idx < self.bev_h) & (y_idx >= 0) & (y_idx < self.bev_w) & (world_z >= self.z_min) & (world_z <= self.z_max))
        linear_idx = x_idx * self.bev_w + y_idx

        feat_vol = context.unsqueeze(3) * depth_prob.unsqueeze(2)
        bev = feat_2d.new_zeros(B, self.context_c, self.bev_h * self.bev_w, dtype=torch.float32)
        counts = feat_2d.new_zeros(B, 1, self.bev_h * self.bev_w, dtype=torch.float32)
        for b in range(B):
            idx_b = linear_idx[b].reshape(-1)
            valid_b = valid[b].reshape(-1)
            if not valid_b.any():
                continue
            feat_b = feat_vol[b].permute(1, 0, 2, 3, 4).reshape(self.context_c, -1)
            idx_valid = idx_b[valid_b]
            feat_valid = feat_b[:, valid_b]
            bev[b].scatter_add_(1, idx_valid.unsqueeze(0).expand(self.context_c, -1), feat_valid)
            counts[b].scatter_add_(1, idx_valid.unsqueeze(0), torch.ones(1, idx_valid.numel(), device=feat_2d.device, dtype=torch.float32))
        bev = bev / counts.clamp(min=1.0)
        return torch.nan_to_num(bev.reshape(B, self.context_c, self.bev_h, self.bev_w), nan=0.0, posinf=0.0, neginf=0.0)


class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c, base_c=96, groups=8):
        super().__init__()
        self.stem = nn.Sequential(ConvGNAct(in_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.down1 = nn.Sequential(ResidualBlock2d(base_c, base_c * 2, 2, groups), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.down2 = nn.Sequential(ResidualBlock2d(base_c * 2, base_c * 4, 2, groups), ResidualBlock2d(base_c * 4, base_c * 4, 1, groups))
        self.aspp = ASPP2d(base_c * 4, base_c * 4, (1,3,6), groups)
        self.up1 = nn.Sequential(ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, 3, 1, 1, groups, True), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.up0 = nn.Sequential(ConvGNAct(base_c * 2 + base_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.head = nn.Conv2d(base_c, 1, 1)
    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)


class MultiCamBEVv6DINOv2LSSClean(nn.Module):
    def __init__(self, num_rover_classes, rover_emb_dim=8, rover_cond_dim=8, n_cameras=4, freeze_backbone=False, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', backbone_out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, context_dim=80, depth_bins=24, depth_min=1.0, depth_max=80.0, world_z_min=-2.0, world_z_max=4.5, bev_base_channels=96, bev_gn_groups=8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _DINOv2MultiScaleBackbone(hub_repo, backbone_name, backbone_out_dim, patch_size, tap_layers, neck_dim, bev_gn_groups)
        self.view_transform = LSSViewTransform2D(neck_dim, context_dim, depth_bins, depth_min, depth_max, BEV_H, BEV_W, BEV_RES, X_RANGE, Y_RANGE, world_z_min, world_z_max, bev_gn_groups)
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        self.rover_mlp = nn.Sequential(nn.Linear(rover_emb_dim, 16), nn.ReLU(inplace=True), nn.Linear(16, rover_cond_dim), nn.ReLU(inplace=True))
        self.bev_decoder = StrongBEVEncoderDecoder(context_dim + rover_cond_dim, bev_base_channels, bev_gn_groups)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
def build_model():
    return MultiCamBEVv6DINOv2LSSClean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        hub_repo=cfg['hub_repo'],
        backbone_name=cfg['backbone_name'],
        backbone_out_dim=cfg['backbone_out_dim'],
        patch_size=cfg['patch_size'],
        tap_layers=cfg['tap_layers'],
        neck_dim=cfg['neck_dim'],
        context_dim=cfg['context_dim'],
        depth_bins=cfg['depth_bins'],
        depth_min=cfg['depth_min'],
        depth_max=cfg['depth_max'],
        world_z_min=cfg['world_z_min'],
        world_z_max=cfg['world_z_max'],
        bev_base_channels=cfg['bev_base_channels'],
        bev_gn_groups=cfg['bev_gn_groups'],
    ).to(device)


def load_model_from_ckpt(ckpt_path: Path, use_ema_if_present: bool):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model = build_model()
    state = ckpt['ema'] if (use_ema_if_present and 'ema' in ckpt) else ckpt['model']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print('loaded from:', ckpt_path.name)
    print('using state:', 'ema' if (use_ema_if_present and 'ema' in ckpt) else 'model')
    print('missing:', len(missing), 'unexpected:', len(unexpected))
    return model


In [ ]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']
    model.eval()
    probs_list, gt_list = [], []
    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)
    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


def threshold_sweep_from_cached_probs(probs, gt, thresholds):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    valid = gt != 255
    gt_b = ((gt == 1) & valid).float()
    for i, t in enumerate(thresholds):
        pred = ((probs > t) & valid).float()
        inter[i] = (pred * gt_b).sum().item()
        union[i] = ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')
    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


In [ ]:
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], rover_vocab=rover_vocab)
ds_test = BEVDatasetV4Clean(test_info, mode='test', img_hw=cfg['img_hw'], rover_vocab=rover_vocab)

loader_val = DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
loader_test = DataLoader(ds_test, batch_size=1, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))

print('val:', len(ds_val), 'test:', len(ds_test))


In [ ]:
thresholds = [round(x, 2) for x in np.arange(0.20, 0.82, 0.02)]
results = []

for candidate_name, ckpt_path, use_ema in [
    ('best', BEST_PT, False),
    ('ema_best', EMA_BEST_PT, True),
]:
    cleanup_cuda()
    model = load_model_from_ckpt(ckpt_path, use_ema_if_present=use_ema)
    val_probs, val_gt = collect_val_probs(model, loader_val, OUT_DIR / f'val_probs_{candidate_name}.pt')
    iou_by_t = threshold_sweep_from_cached_probs(val_probs, val_gt, thresholds)
    best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
    sweep_df = pd.DataFrame({'threshold': list(iou_by_t.keys()), 'iou': list(iou_by_t.values())})
    sweep_df.to_csv(OUT_DIR / f'threshold_sweep_{candidate_name}.csv', index=False)
    print(candidate_name, 'best_t =', best_t, 'best_iou =', best_iou)
    display(sweep_df.sort_values('iou', ascending=False).head(10))
    results.append({'candidate': candidate_name, 'best_t': float(best_t), 'best_iou': float(best_iou)})
    del model
    cleanup_cuda()

results_df = pd.DataFrame(results)
results_df['ema_priority'] = (results_df['candidate'] == 'ema_best').astype(int)
results_df = results_df.sort_values(['best_iou', 'ema_priority'], ascending=[False, False]).reset_index(drop=True)
display(results_df)


In [ ]:
selected_name = results_df.iloc[0]['candidate']
selected_threshold = float(results_df.iloc[0]['best_t'])
selected_ckpt = BEST_PT if selected_name == 'best' else EMA_BEST_PT
selected_use_ema = (selected_name == 'ema_best')

cleanup_cuda()
model = load_model_from_ckpt(selected_ckpt, use_ema_if_present=selected_use_ema)
test_probs = collect_test_probs(model, loader_test, OUT_DIR / f'test_probs_{selected_name}.pt')
print('selected candidate:', selected_name)
print('selected threshold:', selected_threshold)
print('test_probs shape:', tuple(test_probs.shape))
del model
cleanup_cuda()


In [ ]:
def pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'


def make_submission_from_probs(test_probs: torch.Tensor, threshold: float, tag: str):
    thr_tag = f'{threshold:.2f}'.replace('.', 'p')
    pred_dir = OUT_DIR / f'predicted_static_grids_{tag}_{thr_tag}'
    pred_dir.mkdir(parents=True, exist_ok=True)

    for p in pred_dir.glob('*.npy'):
        p.unlink()

    preds = (test_probs > threshold).numpy().astype(np.int32)
    assert len(preds) == len(test_info), (len(preds), len(test_info))

    for i, row in test_info.iterrows():
        np.save(pred_dir / pred_name_from_row(row), preds[i].reshape(1, BEV_H, BEV_W))

    zip_path = OUT_DIR / f'submission_{tag}_t_{thr_tag}.zip'
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(zip_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    result = {
        'threshold': float(threshold),
        'zip_path': str(zip_path.resolve()),
        'size_mb': round(zip_path.stat().st_size / 1e6, 3),
        'sha256': sha,
    }
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


In [ ]:
submit_thresholds = sorted(set([
    round(selected_threshold - 0.04, 2),
    round(selected_threshold - 0.02, 2),
    round(selected_threshold, 2),
    round(selected_threshold + 0.02, 2),
    round(selected_threshold + 0.04, 2),
]))
submit_thresholds = [t for t in submit_thresholds if 0.05 <= t <= 0.95]
print('submit_thresholds:', submit_thresholds)

submission_results = []
for t in submit_thresholds:
    print('\n' + '=' * 100)
    print(f'building submission for threshold={t:.2f}')
    submission_results.append(make_submission_from_probs(test_probs, t, selected_name))

submission_results_df = pd.DataFrame(submission_results)
display(submission_results_df)


In [ ]:
bundle_path = OUT_DIR / 'all_submissions_bundle.zip'
if bundle_path.exists():
    bundle_path.unlink()

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    manifest = []
    for row in submission_results:
        p = Path(row['zip_path'])
        zf.write(p, arcname=p.name)
        manifest.append(row)
    zf.writestr('manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

print('bundle:', bundle_path)
print('bundle size_mb:', round(bundle_path.stat().st_size / 1e6, 3))
